# R1-U1 — pinned pretrained models and the frozen WavLM extractor: Colab verification

Use a **GPU runtime** (T4). Run all cells top to bottom. Needs internet (Hugging Face) but not the dataset. About 1.2 GB is downloaded.

What it does:
1. unpacks `src/`, `scripts/`, `tests/` from `echoproof_sync.zip` (never touches `data/` or `reports/`);
2. runs `tests/test_r1_u1_wavlm.py` (tiny random WavLM, no download);
3. downloads WavLM-Base-Plus **at the pinned commit**, checks the snapshot is exactly that commit, records the weights SHA-256,
   and reads the preprocessing/config values; also records the AST config and preprocessor at its pinned commit (no AST weights);
4. on the real model checks: frozen / head-only trainable, single vs batched parity for full 4 s crops and short clips, order with
   mixed lengths, and (informational) why zero-padded batches are avoided;
5. measures extraction speed and memory (planning input for R2-U1);
6. writes `reports/r1/r1_u1_colab_<timestamp>.json`.

In [1]:
import os, sys, json, subprocess, zipfile, time, hashlib, platform
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

ROOT = Path(os.environ.get("ECHOPROOF_ROOT", "/content/drive/MyDrive/PAI&AIES Project"))
assert ROOT.exists(), f"project folder not found: {ROOT}"

SOURCES = {"pretrained.py": ROOT / "src/echoproof/features/pretrained.py", "test_r1_u1_wavlm.py": ROOT / "tests/test_r1_u1_wavlm.py"}
missing = [k for k, p in SOURCES.items() if not p.exists()]
assert not missing, f"missing in the sync zip: {missing} - rebuild echoproof_sync.zip and upload it"
SHA = {k: hashlib.sha256(p.read_bytes()).hexdigest() for k, p in SOURCES.items()}
print(json.dumps({k: v[:12] for k, v in SHA.items()}, indent=2))

Mounted at /content/drive
{
  "pretrained.py": "aec607d2db80",
  "test_r1_u1_wavlm.py": "eceaa42821b8"
}


In [2]:
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
sys.dont_write_bytecode = True
sys.path.insert(0, str(ROOT / "src"))

import numpy, torch, transformers, huggingface_hub
CUDA = torch.cuda.is_available()
DEVICE = "cuda" if CUDA else "cpu"
ENV = {"python": platform.python_version(), "torch": torch.__version__, "transformers": transformers.__version__,
       "huggingface_hub": huggingface_hub.__version__, "numpy": numpy.__version__, "cuda": CUDA,
       "gpu": torch.cuda.get_device_name(0) if CUDA else None}
print(json.dumps(ENV, indent=2))
if not CUDA:
    print("WARNING: no GPU - timing will not represent a T4. Switch Runtime > Change runtime type > T4 GPU.")

{
  "python": "3.13.16",
  "torch": "2.11.0+cu130",
  "transformers": "5.22.0",
  "huggingface_hub": "1.33.0",
  "numpy": "2.4.6",
  "cuda": true,
  "gpu": "Tesla T4"
}


## 1. Unit tests on this runtime (tiny random WavLM, offline)

In [3]:
proc = subprocess.run([sys.executable, "-m", "pytest", "tests/test_r1_u1_wavlm.py", "-q", "-p", "no:cacheprovider"],
                      cwd=ROOT, env={**os.environ, "PYTHONPATH": str(ROOT / "src")}, capture_output=True, text=True)
print(proc.stdout[-2500:] + proc.stderr[-1500:])
UNIT_TESTS_PASSED = proc.returncode == 0
PYTEST_SUMMARY = next((l.strip() for l in reversed(proc.stdout.splitlines()) if " passed" in l or " failed" in l or " error" in l), "no summary line")
print("unit tests:", "PASSED" if UNIT_TESTS_PASSED else "FAILED", "|", PYTEST_SUMMARY)

.............                                                            [100%]
13 passed in 25.28s

unit tests: PASSED | 13 passed in 25.28s


## 2. Pinned downloads and what the checkpoints say

In [4]:
from echoproof.features.pretrained import PRETRAINED, file_sha256, pinned_snapshot

PINNED = {}
for key, files in (("wavlm", None), ("ast", ["config.json", "preprocessor_config.json"])):   # AST weights are R1-U2's concern
    spec = PRETRAINED[key]
    path = pinned_snapshot(key, files=files)
    cfg = json.loads((path / "config.json").read_text())
    pre = json.loads((path / "preprocessor_config.json").read_text())
    entry = {"repo_id": spec["repo_id"], "revision": spec["revision"], "snapshot_dir": path.name, "license": spec["license"],
             "license_status": spec["license_status"], "config": {k: cfg.get(k) for k in ("model_type", "hidden_size", "num_hidden_layers", "feat_extract_norm", "do_stable_layer_norm", "conv_stride", "conv_kernel")},
             "preprocessor": pre if key == "wavlm" else {k: pre.get(k) for k in ("feature_extractor_type", "sampling_rate", "num_mel_bins", "max_length", "do_normalize", "mean", "std")}}
    weights = path / spec["weights_file"]
    if weights.exists():
        entry["weights_sha256"] = file_sha256(weights)
        entry["weights_bytes"] = weights.stat().st_size
        entry["recorded_sha256_matches"] = (entry["weights_sha256"] == spec["weights_sha256"]) if spec["weights_sha256"] else "none recorded yet - copy this value into PRETRAINED"
    expected = spec["expected_preprocessing"]
    entry["preprocessing_matches_expectation"] = {k: (pre.get(k) == v) for k, v in expected.items()}
    PINNED[key] = entry
    print(json.dumps({k: v for k, v in entry.items() if k != "preprocessor"}, indent=1)[:1800])
print("WavLM preprocessor_config:", json.dumps(PINNED["wavlm"]["preprocessor"]))

# informational: is the pinned commit still what 'main' points to?
for key in PRETRAINED:
    head = huggingface_hub.HfApi().model_info(PRETRAINED[key]["repo_id"]).sha
    PINNED[key]["main_head_now"] = head
    print(key, "pinned == main head:", head == PRETRAINED[key]["revision"])

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

{
 "repo_id": "microsoft/wavlm-base-plus",
 "revision": "4c66d4806a428f2e922ccfa1a962776e232d487b",
 "snapshot_dir": "4c66d4806a428f2e922ccfa1a962776e232d487b",
 "license": null,
 "license_status": "not declared in the Hugging Face metadata; confirm the terms of the WavLM source (microsoft/unilm) before redistributing weights",
 "config": {
  "model_type": "wavlm",
  "hidden_size": 768,
  "num_hidden_layers": 12,
  "feat_extract_norm": "group",
  "do_stable_layer_norm": false,
  "conv_stride": [
   5,
   2,
   2,
   2,
   2,
   2,
   2
  ],
  "conv_kernel": [
   10,
   3,
   3,
   3,
   3,
   2,
   2
  ]
 },
 "weights_sha256": "3bb273a6ace99408b50cfc81afdbb7ef2de02da2eab0234e18db608ce692fe51",
 "weights_bytes": 377617425,
 "recorded_sha256_matches": true,
 "preprocessing_matches_expectation": {
  "sampling_rate": true,
  "do_normalize": true,
  "feature_size": true
 }
}


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

{
 "repo_id": "MIT/ast-finetuned-audioset-10-10-0.4593",
 "revision": "f826b80d28226b62986cc218e5cec390b1096902",
 "snapshot_dir": "f826b80d28226b62986cc218e5cec390b1096902",
 "license": "bsd-3-clause",
 "license_status": "declared in the Hugging Face model card metadata",
 "config": {
  "model_type": "audio-spectrogram-transformer",
  "hidden_size": 768,
  "num_hidden_layers": 12,
  "feat_extract_norm": null,
  "do_stable_layer_norm": null,
  "conv_stride": null,
  "conv_kernel": null
 },
 "preprocessing_matches_expectation": {
  "sampling_rate": true
 }
}
WavLM preprocessor_config: {"do_normalize": false, "feature_extractor_type": "Wav2Vec2FeatureExtractor", "feature_size": 1, "padding_side": "right", "padding_value": 0.0, "return_attention_mask": true, "sampling_rate": 16000}
wavlm pinned == main head: True
ast pinned == main head: True


## 3. Real-model checks: frozen, head-only trainable, single vs batched parity

In [5]:
import numpy as np
from torch import nn
from echoproof.features.pretrained import WavLMExtractor, list_trainable, pool_hidden

TOL = 1e-4
ex = WavLMExtractor.from_pinned(device=DEVICE, batch_size=16)
DESCRIBE = ex.describe()
print(json.dumps(DESCRIBE, indent=1))

rng = np.random.default_rng(0)
def clips(lengths):
    return [(0.3 * np.sin(2 * np.pi * rng.uniform(100, 3000) * np.arange(n) / 16000) + 0.05 * rng.standard_normal(n)).astype(np.float32) for n in lengths]

def worst(a, b):
    return float(np.max(np.abs(a - b)))

RESULTS = {}
ex.train()
RESULTS["frozen"] = {"trainable_names": list_trainable(ex), "in_eval_after_train_call": (not ex.training) and (not ex.encoder.training)}

class WithHead(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Sequential(nn.Linear(encoder.dim, 128), nn.GELU(), nn.Dropout(0.2), nn.Linear(128, 2))
model = WithHead(ex)
model.train()
RESULTS["head_only"] = {"trainable": list_trainable(model), "encoder_in_eval_under_model_train": not model.encoder.encoder.training, "head_in_train": model.head.training}

parity = {}
for label, length, count in (("full_4s_crop", 64000, 8), ("2.3s_clip", 37000, 5), ("1s_clip", 16000, 5), ("0.5s_clip", 8000, 5)):
    batch = clips([length] * count)
    row = {}
    for pooling in ("mean", "mean_std"):
        together = ex.embed(batch, pooling)
        row[pooling] = max(worst(ex.embed([w], pooling)[0], together[i]) for i, w in enumerate(batch))
        row[pooling + "_shape"] = list(together.shape)
    parity[label] = row
RESULTS["single_vs_batched_max_abs_diff"] = parity

mixed = clips([64000, 16000, 64000, 37000, 16000, 8000, 64000])
together = ex.embed(mixed, "mean_std")
RESULTS["mixed_lengths_order_max_abs_diff"] = max(worst(ex.embed([w], "mean_std")[0], together[i]) for i, w in enumerate(mixed))
RESULTS["batch_size_1_vs_16_max_abs_diff"] = worst(WavLMExtractor.from_pinned(device=DEVICE, batch_size=1).embed(mixed[:4]), ex.embed(mixed[:4]))
RESULTS["repeat_run_max_abs_diff"] = worst(ex.embed(mixed[:3]), ex.embed(mixed[:3]))

# informational: why batches are never zero-padded
short, long = clips([16000])[0], clips([64000])[0]
alone = ex.embed([short])[0]
x = torch.zeros(2, 64000, device=DEVICE)
x[0, :16000] = torch.from_numpy(short).to(DEVICE)
x[1] = torch.from_numpy(long).to(DEVICE)
with torch.inference_mode():
    hidden = ex.encoder(x).last_hidden_state
padded = pool_hidden(hidden, torch.tensor([ex.hidden_length(16000), ex.hidden_length(64000)]))[0].cpu().numpy()
RESULTS["zero_padding_would_change_embedding_by"] = worst(alone, padded)

# informational: CPU vs GPU, relevant if embeddings are cached on one device and compared on another
if CUDA:
    cpu_ex = WavLMExtractor.from_pinned(device="cpu", batch_size=4)
    RESULTS["cpu_vs_gpu_max_abs_diff"] = worst(cpu_ex.embed(mixed[:3], "mean_std"), ex.embed(mixed[:3], "mean_std"))
    del cpu_ex
print(json.dumps(RESULTS, indent=1))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

{
 "hidden_size": 768,
 "parameters": 94381936,
 "trainable_parameters": 0,
 "training_mode": false,
 "batch_size": 16,
 "feature_extractor": "Wav2Vec2FeatureExtractor",
 "do_normalize": false
}


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

{
 "frozen": {
  "trainable_names": [],
  "in_eval_after_train_call": true
 },
 "head_only": {
  "trainable": [
   "head.0.bias",
   "head.0.weight",
   "head.3.bias",
   "head.3.weight"
  ],
  "encoder_in_eval_under_model_train": true,
  "head_in_train": true
 },
 "single_vs_batched_max_abs_diff": {
  "full_4s_crop": {
   "mean": 8.642673492431641e-07,
   "mean_shape": [
    8,
    768
   ],
   "mean_std": 8.642673492431641e-07,
   "mean_std_shape": [
    8,
    1536
   ]
  },
  "2.3s_clip": {
   "mean": 5.960464477539062e-07,
   "mean_shape": [
    5,
    768
   ],
   "mean_std": 6.705522537231445e-07,
   "mean_std_shape": [
    5,
    1536
   ]
  },
  "1s_clip": {
   "mean": 8.940696716308594e-07,
   "mean_shape": [
    5,
    768
   ],
   "mean_std": 9.08970832824707e-07,
   "mean_std_shape": [
    5,
    1536
   ]
  },
  "0.5s_clip": {
   "mean": 1.6689300537109375e-06,
   "mean_shape": [
    5,
    768
   ],
   "mean_std": 1.6689300537109375e-06,
   "mean_std_shape": [
    5,
   

## 4. Extraction speed and memory (planning input for R2-U1)

In [6]:
bench = clips([64000] * 64)
ex.embed(bench[:16])                              # warm-up
if CUDA:
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
t0 = time.perf_counter()
ex.embed(bench, "mean_std")
if CUDA:
    torch.cuda.synchronize()
seconds = time.perf_counter() - t0
TIMING = {"device": DEVICE, "clips": len(bench), "ms_per_4s_clip": round(seconds / len(bench) * 1000, 1),
          "clips_per_second": round(len(bench) / seconds, 1), "peak_vram_mb": round(torch.cuda.max_memory_allocated() / 2**20, 0) if CUDA else None,
          "projected_minutes_for_60000_chunks": round(60000 * seconds / len(bench) / 60, 1)}
print(json.dumps(TIMING, indent=1))

{
 "device": "cuda",
 "clips": 64,
 "ms_per_4s_clip": 27.7,
 "clips_per_second": 36.1,
 "peak_vram_mb": 1186.0,
 "projected_minutes_for_60000_chunks": 27.7
}


## 5. Verdict and evidence

In [7]:
problems = []
if not UNIT_TESTS_PASSED:
    problems.append("unit tests failed")
for key, entry in PINNED.items():
    if entry["snapshot_dir"] != PRETRAINED[key]["revision"]:
        problems.append(f"{key}: snapshot is not the pinned commit")
    if entry.get("recorded_sha256_matches") is False:
        problems.append(f"{key}: weights SHA-256 differs from the recorded value")
    bad = [k for k, ok in entry["preprocessing_matches_expectation"].items() if not ok]
    if bad:
        problems.append(f"{key}: preprocessing differs from the expectation in PRETRAINED for {bad} (update the expectation deliberately, or investigate)")
if RESULTS["frozen"]["trainable_names"] or not RESULTS["frozen"]["in_eval_after_train_call"]:
    problems.append("encoder is not frozen / not in eval mode")
if any(n.startswith("encoder.") for n in RESULTS["head_only"]["trainable"]) or not RESULTS["head_only"]["trainable"] or not RESULTS["head_only"]["encoder_in_eval_under_model_train"]:
    problems.append("trainable parameters are not exactly the head")
for label, row in RESULTS["single_vs_batched_max_abs_diff"].items():
    for pooling in ("mean", "mean_std"):
        if row[pooling] > TOL:
            problems.append(f"{label}/{pooling}: single vs batched differs by {row[pooling]:.2e} (> {TOL})")
for name in ("mixed_lengths_order_max_abs_diff", "batch_size_1_vs_16_max_abs_diff"):
    if RESULTS[name] > TOL:
        problems.append(f"{name} = {RESULTS[name]:.2e} (> {TOL})")
if RESULTS["repeat_run_max_abs_diff"] > 1e-6:
    problems.append("repeated runs differ")

verdict = {"task": "R1-U1", "verified": not problems and CUDA, "gpu_run": CUDA, "problems": problems, "tolerance": TOL, "pytest_summary": PYTEST_SUMMARY}
evidence = {"schema_version": 1, "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "environment": ENV, "source_sha256": SHA,
            "verdict": verdict, "pinned": PINNED, "extractor": DESCRIBE, "checks": RESULTS, "timing": TIMING}
out_dir = ROOT / "reports" / "r1"
out_dir.mkdir(parents=True, exist_ok=True)
path = out_dir / f"r1_u1_colab_{time.strftime('%Y%m%dT%H%M%SZ', time.gmtime())}.json"
path.write_text(json.dumps(evidence, indent=2, sort_keys=True, default=str), encoding="utf-8")
print(json.dumps(verdict, indent=2))
print("\nevidence written to:", path)

{
  "task": "R1-U1",
  "verified": true,
  "gpu_run": true,
  "problems": [],
  "tolerance": 0.0001,
  "pytest_summary": "13 passed in 25.28s"
}

evidence written to: /content/drive/MyDrive/PAI&AIES Project/reports/r1/r1_u1_colab_20261104T135216Z.json
